# IMPORTING

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# relative to the notebooks folder
DATA_FOLDER = Path("../.csv/2025-oct to 2026-oct")

def load(name):
    df = pd.read_csv(DATA_FOLDER / f"{name}.csv", dtype_backend="numpy_nullable")
    for col in df.columns:
        if col.endswith("_at"):
            df[col] = pd.to_datetime(df[col], utc=True)
    return df

prs = load("pull_requests")
issues = load("issues")
reviews = load("reviews")
review_comments = load("review_comments")
comments = load("comments")
users = load("users")

items = pd.concat(
    [prs.assign(type="pull_request"), issues.assign(type="issue")],
    ignore_index=True,
)

all_dfs = [items, prs, issues, reviews, review_comments, comments, users]

meta = json.loads((DATA_FOLDER / "dataset.json").read_text())
meta

{'owner': 'godotengine',
 'repo': 'godot',
 'from': '2025-10-18T00:00:00+00:00',
 'to': '2026-10-18T00:00:00+00:00',
 'sample_size': None,
 'seed': 42,
 'include_issues': True,
 'include_pull_requests': True,
 'fetch_user_profiles': True,
 'exported_at': '2026-10-23T16:03:52.5563767+00:00'}

In [2]:
for df in all_dfs:
    print(f"{df.shape}")

(12747, 27)
(7533, 22)
(5214, 16)
(17409, 9)
(16278, 11)
(37904, 9)
(4485, 17)


# BOT FILTERING

In [3]:
# from dotenv import load_dotenv
# import os

# load_dotenv()
# ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")

In [4]:
# from rabbit_ng import run_rabbit

# for result in run_rabbit(
#     contributors=list(users["login"]),
#     api_key=ACCESS_TOKEN,
#     min_events=5,
#     min_confidence=1.0,
#     max_queries=3,
#     no_wait=False,
# ):
#     if result.user_type == "Bot":
#         print(f"{result.contributor}: (Confidence: {result.confidence})")

In [5]:
import re

def is_bot(login: pd.Series, account_type) -> pd.Series:
    return (
        (account_type == "Bot")
        | login.str.contains("Bot", na=False, flags=re.IGNORECASE, regex=True)
    )

SUSPECTED_BOTS = set()

for df in all_dfs:  # adds an is_bot column to every df and fills the SUSPECTED_BOTS set
    login_col = "reviewer" if df is reviews else "login" if df is users else "author"
    sus = is_bot(df[login_col], df["type" if df is users else f"{login_col}_type"])
    SUSPECTED_BOTS.update(df.loc[sus, login_col].dropna())

SUSPECTED_BOTS

{'CodeBot-git',
 'Copilot',
 'DungenRobot',
 'Hamood-bot',
 'Noobot9k',
 'RobotBuddy',
 'Roboticy3',
 'SleepingRobot',
 'bamiebot-maker',
 'benbot',
 'copilot-pull-request-reviewer[bot]',
 'dependabot[bot]'}

In [6]:
for login in SUSPECTED_BOTS:
    try:
        print(users.loc[users["login"] == login, "url"].iloc[0])
    except:
        pass

https://github.com/RobotBuddy
https://github.com/CodeBot-git
https://github.com/SleepingRobot
https://github.com/bamiebot-maker
https://github.com/benbot
https://github.com/Hamood-bot
https://github.com/Noobot9k
https://github.com/apps/dependabot
https://github.com/DungenRobot
https://github.com/apps/copilot-pull-request-reviewer
https://github.com/Roboticy3


In [7]:
# After inspecting flagged accounts, only the following were determined as bots
BOTS = {
    "copilot-pull-request-reviewer[bot]",
    "dependabot[bot]",
    "Copilot",
}

for df in all_dfs:
    login_col = "reviewer" if df is reviews else "login" if df is users else "author"
    df["is_bot"] = df[login_col].isin(BOTS)

# ALIAS FILTERING

### Normalising the data

In [8]:
import re

def normalize_login(login: str):
    s = re.sub(r"[-_.]", "", login.lower())  # drop separators
    return re.sub(r"\d+$", "", s) or s       # drop trailing digits

def normalize_link(link: str):
    s = re.sub(r"^https?://", "", link.strip().lower())
    return re.sub(r"^www\.", "", s).rstrip("/")

norm_users = pd.DataFrame()

norm_users["login"] = users[~users["is_bot"]]["login"]
norm_users["norm_login"] = users["login"].map(normalize_login)
norm_users["norm_email"] = users["email"].fillna("").str.strip().str.lower()
norm_users["norm_blog"] = users["blog"].fillna("").map(normalize_link)

norm_users

,login,norm_login,norm_email,norm_blog
0,014-code,014code,,014-code.github.io/014.github.io
1,0MAD03,0mad,,
2,0TheJCM0,0thejcm,,
3,0ttoman,0ttoman,,
4,0x0ACB,0x0acb,,
...,...,...,...,...
4480,Zybyte85,zybyte,,
4481,Zylann,zylann,,zylannprods.fr/fr
4482,zzag,zzag,contact@vladzahorodnii.com,blog.vladzahorodnii.com
4483,zzimbaa,zzimbaa,,


### Defining filtering methods

In [9]:
from difflib import SequenceMatcher

FILTER_LOGIN = True
FILTER_BLOG = True
FILTER_EMAIL = True

NAME_THRESHOLD = 0.85  # 0..1, how alike two names/logins must be
MIN_LENGTH = 4         # ignore very short names/logins, they match too easily

def is_similar(x: str, y: str):
    if len(x) < MIN_LENGTH or len(y) < MIN_LENGTH:
        return False
    m = SequenceMatcher(None, x, y)
    # the two quick ratios are cheap upper bounds that rule out most pairs before the slow exact ratio
    return m.real_quick_ratio() >= NAME_THRESHOLD and m.quick_ratio() >= NAME_THRESHOLD and m.ratio() >= NAME_THRESHOLD

# the 2 measures
def similar_name(a, b):
    return (FILTER_LOGIN and is_similar(a.norm_login, b.norm_login))

def same_contact(a, b):
    def match(x, y) -> bool: 
        return x != "" and x == y
    return (FILTER_BLOG and match(a.norm_blog, b.norm_blog)) or (FILTER_EMAIL and match(a.norm_email, b.norm_email))

# wrapper for the 2 measures
def is_alias(a, b):
    return similar_name(a, b) or same_contact(a, b)

### Filtering the data

In [10]:
from itertools import combinations

SUSPECTED_ALIASES = {}

def add_alias(a, b):
    main_a = SUSPECTED_ALIASES.get(a, a)
    main_b = SUSPECTED_ALIASES.get(b, b)
    if main_a == main_b: return  # already the same person

    # everything that pointed to main_a now points to main_b, then main_a itself does too
    for alt, main in SUSPECTED_ALIASES.items():
        if main == main_a:
            SUSPECTED_ALIASES[alt] = main_b
    SUSPECTED_ALIASES[main_a] = main_b

# populate ALIASES
for a, b in combinations(norm_users.itertuples(index=False), 2):
    if is_alias(a, b): add_alias(a.login, b.login)

SUSPECTED_ALIASES

{'a-johnston': 'ijohnston1911',
 'aljohnston112': 'ijohnston1911',
 'Aaron1318111': 'aaronp64',
 'Alex2782': 'alex44444444',
 'cFonzy': 'fonzy1243',
 'chrom007': 'jchrom',
 'dit-zy': 'Ditzy1571',
 'dugramen': 'Edugamen',
 'elix22': 'elixs',
 'fighter-435': 'Fighter19',
 'fire': 'Fire551',
 'InkRobert': 'rogerty',
 'jaideng123': 'Jaydeng75',
 'JAMES-535': 'jams3223',
 'James103': 'jams3223',
 'JamesK89': 'jams3223',
 'kisg': 'konczg',
 'krobert': 'rogerty',
 'Robert-K': 'rogerty',
 'Lizard787': 'Splizard',
 'Nova840': 'unova-01',
 'peter1745': 'PeterD23',
 'Peter2210': 'PeterD23',
 'RobertY05': 'rogerty',
 'shadow64': 'Shadow705',
 'Silver1063': 'Silver1078682',
 'Tamir198': 'Tlamir',
 'UweSchneider': 'UweSchneiderGmbH',
 'warren': 'Warren1001'}

In [11]:
for key, value in SUSPECTED_ALIASES.items():
    print(f"{users.loc[users['login'] == key, 'url'].iloc[0]} : {users.loc[users['login'] == value, 'url'].iloc[0]}")

https://github.com/a-johnston : https://github.com/ijohnston1911
https://github.com/aljohnston112 : https://github.com/ijohnston1911
https://github.com/Aaron1318111 : https://github.com/aaronp64
https://github.com/Alex2782 : https://github.com/alex44444444
https://github.com/cFonzy : https://github.com/fonzy1243
https://github.com/chrom007 : https://github.com/jchrom
https://github.com/dit-zy : https://github.com/Ditzy1571
https://github.com/dugramen : https://github.com/Edugamen
https://github.com/elix22 : https://github.com/elixs
https://github.com/fighter-435 : https://github.com/Fighter19
https://github.com/fire : https://github.com/Fire551
https://github.com/InkRobert : https://github.com/rogerty
https://github.com/jaideng123 : https://github.com/Jaydeng75
https://github.com/JAMES-535 : https://github.com/jams3223
https://github.com/James103 : https://github.com/jams3223
https://github.com/JamesK89 : https://github.com/jams3223
https://github.com/kisg : https://github.com/konczg
h

In [12]:
# After inspecting flagged accounts, only the following were determined as aliases

ALIASES = {
    "UweSchneider" : "UweSchneiderGmbH"
}

for df in all_dfs:
    login_col = "reviewer" if df is reviews else "login" if df is users else "author"
    df["main_login"] = df[login_col].replace(ALIASES)

# PROCESS-METRICS

## 1. Time to First Response: [6] 
A measurement of how long it takes for another person to respond to an action that requires attention (Like a pull request or issue).

### Applying filters to data

In [13]:
# find the main_login of the author of every item, by item number
author_of = items.set_index("number")["main_login"]

# mark reviews/comments that were made by the author of the item they belong to
reviews["pr_author"] = reviews["pr_number"].map(author_of)
reviews["by_author"] = (reviews["main_login"] == reviews["pr_author"]).fillna(False)
comments["item_author"] = comments["parent_number"].map(author_of)
comments["by_author"] = (comments["main_login"] == comments["item_author"]).fillna(False)

In [14]:
# exclude items that were authored by bots
m1_items = items[~items["is_bot"]].copy()

# exclude reviews/comments that were authored by bots
m1_reviews = reviews[~reviews["is_bot"]].copy()
m1_comments = comments[~comments["is_bot"]].copy()

# exclude reviews/comments that were created by the author of their item
m1_reviews = m1_reviews[~m1_reviews["by_author"]]
m1_comments = m1_comments[~m1_comments["by_author"]]

### Calculating metric

In [15]:
# time from creation until the first review / comment by another person
m1_items["first_review"] = m1_items["number"].map(m1_reviews.groupby("pr_number")["submitted_at"].min()) - m1_items["created_at"]
m1_items["first_comment"] = m1_items["number"].map(m1_comments.groupby("parent_number")["created_at"].min()) - m1_items["created_at"]

# the first response is whichever of the three came first
m1_items["first_response"] = m1_items[["first_review", "first_comment"]].min(axis=1)

m1_items

,number,title,state,author,author_type,created_at,updated_at,closed_at,merged_at,merged,...,type,state_reason,closed_by,assignees,locked,is_bot,main_login,first_review,first_comment,first_response
0,111780,Allow double-clicking icons of non-editable Tr...,closed,aaronfranke,User,2025-10-18 02:47:42+00:00,2026-02-07 12:35:53+00:00,2025-11-10 23:52:49+00:00,2025-11-10 23:52:49+00:00,True,...,pull_request,<NA>,<NA>,<NA>,<NA>,False,aaronfranke,23 days 18:03:17,23 days 18:03:08,23 days 18:03:08
1,111782,Update note for `rendering/rendering_device/vs...,closed,Kaleb-Reid,User,2025-10-18 04:40:00+00:00,2026-01-25 09:39:53+00:00,2025-10-20 23:15:21+00:00,2025-10-20 23:15:21+00:00,True,...,pull_request,<NA>,<NA>,<NA>,<NA>,False,Kaleb-Reid,0 days 00:53:26,2 days 18:35:48,0 days 00:53:26
2,111783,Implement Directional Shadow Splits debug mode...,open,Kaleb-Reid,User,2025-10-18 07:13:08+00:00,2026-06-19 06:55:50+00:00,NaT,NaT,False,...,pull_request,<NA>,<NA>,<NA>,<NA>,False,Kaleb-Reid,0 days 14:39:32,NaT,0 days 14:39:32
3,111784,Add Unit Tests for StreamPeerTLS,open,Sherv01,User,2025-10-18 07:38:50+00:00,2026-02-17 20:10:07+00:00,NaT,NaT,False,...,pull_request,<NA>,<NA>,<NA>,<NA>,False,Sherv01,NaT,0 days 00:25:34,0 days 00:25:34
4,111785,Add C# Bindings for GDExtension types and corr...,open,Ryan-000,User,2025-10-18 09:17:59+00:00,2026-02-17 20:10:06+00:00,NaT,NaT,False,...,pull_request,<NA>,<NA>,<NA>,<NA>,False,Ryan-000,NaT,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12742,124587,C#: released weak GC handle left after a dead-...,open,begezavr2,User,2026-10-17 18:54:54+00:00,2026-10-19 09:13:05+00:00,NaT,NaT,<NA>,...,issue,<NA>,<NA>,<NA>,False,False,begezavr2,NaT,1 days 14:18:02,1 days 14:18:02
12743,124588,extends Node func _ready(): # Enable V-Sy...,closed,mdosman01833759281-commits,User,2026-10-17 19:22:19+00:00,2026-10-17 19:35:22+00:00,2026-10-17 19:34:25+00:00,NaT,<NA>,...,issue,not_planned,rsubtil,<NA>,False,False,mdosman01833759281-commits,NaT,NaT,NaT
12744,124591,"Sticky items hidden while dragging, but still ...",open,AeioMuch,User,2026-10-17 21:47:11+00:00,2026-10-19 08:43:02+00:00,NaT,NaT,<NA>,...,issue,<NA>,<NA>,<NA>,False,False,AeioMuch,NaT,NaT,NaT
12745,124592,Godot refuses to use OpenGL if driver support ...,open,jummy0,User,2026-10-17 22:12:17+00:00,2026-10-19 09:19:36+00:00,NaT,NaT,<NA>,...,issue,<NA>,<NA>,<NA>,False,False,jummy0,NaT,0 days 00:06:56,0 days 00:06:56


### Reporting Data

In [16]:
# automatically drops na values
m1_items["first_response"].groupby(m1_items["type"]).describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
issue,3761,6 days 16:14:21.195692,27 days 03:10:06.373807,0 days 00:00:32,0 days 01:08:26,0 days 06:48:45,1 days 06:15:32,301 days 03:28:27
pull_request,6376,7 days 05:30:21.060696,25 days 16:51:44.945221,0 days 00:00:25,0 days 00:43:22.500000,0 days 05:58:07.500000,1 days 12:03:49.750000,363 days 01:35:23


In [ ]:
no_response = m1_items[m1_items["first_response"].isna()].copy()
no_response["number"].groupby(no_response["type"]).count()

type
issue           1453
pull_request    1156
Name: number, dtype: Int64

In [ ]:
no_response_yes_close = no_response[no_response["closed_at"].notna()]
no_response_yes_close["closing_time"] = no_response_yes_close["closed_at"] - no_response_yes_close["created_at"]
no_response_yes_close["closing_time"].groupby(no_response_yes_close["type"]).describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
issue,709,15 days 16:06:19.077574,36 days 07:49:35.100953,0 days 00:00:13,0 days 12:48:34,2 days 03:53:12,9 days 12:56:06,285 days 00:38:26
pull_request,308,18 days 03:53:13.301948,50 days 20:25:15.479602,0 days 00:00:04,0 days 00:04:51,0 days 01:56:57.500000,2 days 20:13:37.250000,286 days 19:14:00


In [21]:
no_response_yes_close["closing_time"].groupby(no_response_yes_close["type"]).head()

154      0 days 14:48:28
275      0 days 15:19:20
359      0 days 00:00:20
370    223 days 02:45:24
383      1 days 16:12:59
7556     2 days 04:17:04
7558    32 days 00:09:54
7559     0 days 02:25:08
7560     0 days 02:05:30
7563     0 days 14:42:12
Name: closing_time, dtype: timedelta64[us]

https://github.com/godotengine/godot/issues/124578 example 

## Time to Close: [5] 
A measurement of how long it takes for an action like pull request or issue to be closed.